In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from tabulate import tabulate

data = pd.read_csv('geochemical_data_final.csv')
print(data.columns)

# Separate features and response variables
X = data.iloc[:, 4:17]                          # features
Y = data.iloc[:, -1]                            # response variable, geothermal reservoir temperature.
print(f'Features of dataset: {X.columns}')
print(f'Number of compenents in features: {X.shape[1]}')
print(Y.head())

Index(['name', 'manifestation_type', 'geothermal_system', 'id', 't_out', 'pH',
       'ac_carbonate', 'chloride', 'sulfate', 'calcium', 'magnesium', 'sodium',
       'potassium', 'lithium', 'silica', 'δ18O-H2O', 'δD-H2O', 'T_reservoir'],
      dtype='object')
Features of dataset: Index(['t_out', 'pH', 'ac_carbonate', 'chloride', 'sulfate', 'calcium',
       'magnesium', 'sodium', 'potassium', 'lithium', 'silica', 'δ18O-H2O',
       'δD-H2O'],
      dtype='object')
Number of compenents in features: 13
0    166.0
1    174.0
2    159.7
3    223.0
4    218.0
Name: T_reservoir, dtype: float64


In [3]:
### Linear Model - Elastic-Net
# Elastic Net is a regularization technique that combines Lassi (L1) and Ridge (L2) regularization or penalties.
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_squared_log_error, mean_absolute_error
import joblib

x_train_lm, x_test_lm, y_train_log_lm, y_test_log_lm = train_test_split(X, np.log(Y), test_size=0.1, random_state=42)
print(f'Number of samples in training set: {x_train_lm.shape[0]}')

scaler = StandardScaler()
x_train_lm = scaler.fit_transform(x_train_lm)
x_test_lm = scaler.transform(x_test_lm)

# Setting up grid search cross validation for parameters tuning
# esimator: elastic_net model
# param_grid: dictionary of parameters to be tested.
# cv: number of split for cross-validation (5- fold cross-validation: validación cruzada quíntuple)
# scoring or evaluation metrics: mean squeared error is used here, GridSearchCV maximize the score or the metrics.

start_time_lm = time.time()

elastic_net = ElasticNet()
param_lm = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

param_grid_lm = GridSearchCV(
    estimator=elastic_net, 
    param_grid=param_lm, 
    cv=5, 
    scoring='neg_mean_squared_error',
    verbose=1, 
    n_jobs=-1                                                   # n_jobs=-1 utilize all the cores of the computer avalaible to perform the cv and 
)                                                               # parameter tuning in parallel, making the grid search more efficient.

# Perform grid search
param_grid_lm.fit(x_train_lm, y_train_log_lm)                   # fit=fit the model to the training data.

# Get the best model
best_model_lm = param_grid_lm.best_estimator_                   # best_estimator_ attribute of the GridSearchCV object returns the best model.
print(f'Best parameters for Elastic-Net: {best_model_lm}')

y_pred_test_log_lm = best_model_lm.predict(x_test_lm)           # predict the response variable (Geothermal Reservoir Temperature) using the best model.
y_pred_train_log_lm = best_model_lm.predict(x_train_lm)

y_pred_test_lm = np.exp(y_pred_test_log_lm)                     # convert the predicted response variable to the original scale.
y_pred_train_lm = np.exp(y_pred_train_log_lm)
y_test_lm = np.exp(y_test_log_lm)
y_train_lm = np.exp(y_train_log_lm)

end_time_lm = time.time()
training_time_lm = end_time_lm - start_time_lm
print(f'Training time for Elastic-Net: {training_time_lm}')

# Evaluate the model
def mean_relative_squared_error(y_true, y_pred):
    return np.mean(np.square((y_true - y_pred) / y_true))

r2_lm = r2_score(y_test_lm, y_pred_test_lm)
mse_lm = mean_squared_error(y_test_lm, y_pred_test_lm)
mslr_lm = mean_squared_log_error(y_test_lm, y_pred_test_lm)
mae_lm = mean_absolute_error(y_test_lm, y_pred_test_lm)
mrse_lm = mean_relative_squared_error(y_test_lm, y_pred_test_lm)

eval_metrics_lm = {
    'Eval_metrics': ['R2 Score', 'MSE', 'MAE', 'MSLE', 'MRSE', 'Training time'],
    'Elastic-Net Model': [r2_lm, mse_lm, mae_lm, mslr_lm, mrse_lm, training_time_lm]
}

metrics_lm = pd.DataFrame(eval_metrics_lm)
metrics_lm.to_csv('metrics_lm', index=False)

# Save the model
joblib.dump(best_model_lm, 'elastic_net_model.pkl')
print(f'Model saved as elastic_net_model.pkl')

print(tabulate(metrics_lm.round(4), headers='keys', tablefmt='pretty', showindex=False))

Number of samples in training set: 158
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters for Elastic-Net: ElasticNet(alpha=0.1, l1_ratio=0.1)
Training time for Elastic-Net: 2.2600598335266113
Model saved as elastic_net_model.pkl
+---------------+-------------------+
| Eval_metrics  | Elastic-Net Model |
+---------------+-------------------+
|   R2 Score    |      -0.4515      |
|      MSE      |     1018.0681     |
|      MAE      |      25.0095      |
|     MSLE      |      0.0225       |
|     MRSE      |      0.0192       |
| Training time |      2.2601       |
+---------------+-------------------+


In [4]:
### Generalized Linear Model - Tweedie Regression

from sklearn.linear_model import TweedieRegressor

x_train_tweedie, x_test_tweedie, y_train_tweedie, y_test_tweedie = train_test_split(X, Y, test_size=0.1, random_state=42)

tw_regressor = TweedieRegressor(power=1, alpha=0.5, link='log')
tw_regressor.fit(x_train_tweedie, y_train_tweedie)

y_pred_test_tweedie = tw_regressor.predict(x_test_tweedie)
y_pred_train_tweedie = tw_regressor.predict(x_train_tweedie)

r2_tweedie = r2_score(y_test_tweedie, y_pred_test_tweedie)
mse_tweedie = mean_squared_error(y_test_tweedie, y_pred_test_tweedie)
mslr_tweedie = mean_squared_log_error(y_test_tweedie, y_pred_test_tweedie)
mae_tweedie = mean_absolute_error(y_test_tweedie, y_pred_test_tweedie)
mrse_tweedie = mean_relative_squared_error(y_test_tweedie, y_pred_test_tweedie)

eval_metrics_tweedie = {
    'Eval_metrics': ['R2 Score', 'MSE', 'MAE', 'MSLE', 'MRSE'],
    'Tweedie Model': [r2_tweedie, mse_tweedie, mae_tweedie, mslr_tweedie, mrse_tweedie]
}

metrics_tweedie = pd.DataFrame(eval_metrics_tweedie)
metrics_tweedie.to_csv('metrics_tweedie', index=False)

print(tabulate(metrics_tweedie.round(4), headers='keys', tablefmt='pretty', showindex=False))

# Save the model
joblib.dump(tw_regressor, 'tweedie_model.pkl')
print(f'Model saved as tweedie_model.pkl')

+--------------+---------------+
| Eval_metrics | Tweedie Model |
+--------------+---------------+
|   R2 Score   |    -0.1754    |
|     MSE      |   824.4113    |
|     MAE      |    25.8118    |
|     MSLE     |    0.0177     |
|     MRSE     |    0.0168     |
+--------------+---------------+
Model saved as tweedie_model.pkl


/home/chris/venv/cdd/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:295: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


In [10]:
### Simple desicion tree

from sklearn.tree import DecisionTreeRegressor

start_time_dt = time.time()
x_train_dt, x_test_dt, y_train_dt, y_test_dt = train_test_split(X, Y, test_size=0.1, random_state=42)

# Parameters tuning
tree_regressor = DecisionTreeRegressor()
param_grid_dt = {'criterion': ['squared_error'],
            'max_depth': [50, 75, 100],
            'min_samples_split': [5, 10, 25],
            'max_leaf_nodes': [25, 50, 75],
            'min_impurity_decrease': [0, 0.01, 0.1]}


grid_search_cv = GridSearchCV(estimator=tree_regressor, param_grid=param_grid_dt, cv=5, verbose=1, scoring='neg_mean_squared_error')
grid_search_cv.fit(x_train_dt, y_train_dt)
best_model_dt = grid_search_cv.best_estimator_
print(f'Desicion tree regressor best parameters: {best_model_dt}')

y_pred_test_dt = best_model_dt.predict(x_test_dt)
y_pred_train_dt = best_model_dt.predict(x_train_dt)

end_time_dt = time.time()
training_time_dt = end_time_dt - start_time_dt

r2_dt = r2_score(y_test_dt, y_pred_test_dt)
mse_dt = mean_squared_error(y_test_dt, y_pred_test_dt)
mslr_dt = mean_squared_log_error(y_test_dt, y_pred_test_dt)
mae_dt = mean_absolute_error(y_test_dt, y_pred_test_dt)
mrse_dt = mean_relative_squared_error(y_test_dt, y_pred_test_dt)

eval_metrics_dt = {
    'Eval_metrics': ['R2 Score', 'MSE', 'MAE', 'MSLE', 'MRSE', 'Training time'],
    'Elastic-Net Model': [r2_dt, mse_dt, mae_dt, mslr_dt, mrse_dt, training_time_dt]
}

df_metrics_dt = pd.DataFrame(eval_metrics_dt)
df_metrics_dt.to_csv('metrics_dt.csv', index=False)

print(tabulate(df_metrics_dt.round(4), headers='keys', tablefmt='pretty', showindex=False))

joblib.dump(tree_regressor, 'dt_model.joblib')
print("Decision Tree model saved as 'decision_tree_model.joblib'.")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Desicion tree regressor best parameters: DecisionTreeRegressor(max_depth=75, max_leaf_nodes=50,
                      min_impurity_decrease=0.01, min_samples_split=10)
+---------------+-------------------+
| Eval_metrics  | Elastic-Net Model |
+---------------+-------------------+
|   R2 Score    |      0.7287       |
|      MSE      |     190.2565      |
|      MAE      |      8.7935       |
|     MSLE      |      0.0042       |
|     MRSE      |      0.0038       |
| Training time |      1.1994       |
+---------------+-------------------+
Decision Tree model saved as 'decision_tree_model.joblib'.


In [ ]:
### Neural Network implementation (Keras)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import KFold

start_time_k = time.time()

x_train_k, x_test_k, y_train_k, y_test_k = train_test_split(X, Y, test_size=0.2, random_state=36)

scaler = StandardScaler()
x_train_k = scaler.fit_transform(x_train_k)
x_test_k = scaler.transform(x_test_k)

# Define neural networks architecture with LeakyReLU activation

model = Sequential([
    Dense(512, input_dim=x_train_k.shape[1], activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(256, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(1, activation='linear')
])

# Compile the model 
model.compile(optimizer=Adam(learning_rate=0.0005), loss='mean_squared_error', metrics=['mean_absolute_error'])

# St up callbacks for early stopping and learning rate reduction
early_stop = EarlyStopping(monitor='val_loss',patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=10, min_lr=1e-6)

# Train the model
training = model.fit(x_train_k, y_train_k, epochs=1500, validation_split=0.2, batch_size=16,
                     verbose=2, callbacks=[early_stop, reduce_lr])

end_time_k = time.time()

# Save the model
model.save('keras_nn_model.h5')
print("Model saved to 'keras_nn_model.h5'.")

# Predict the model
y_pred_test_k = model.predict(x_test_k)
y_pred_train_k = model.predict(x_train_k)

# Return the responses to the origina scale


Y_pred_test_nn = np.squeeze(y_pred_test_k)

training_time_k = end_time_k - start_time_k

# Evaluate the model
def mean_relative_squared_error(y_true, y_pred_test):
    return np.mean(((y_true - y_pred_test)/y_true)**2)

r2_k = r2_score(y_test_k, y_pred_test_k)
mse_k = mean_squared_error(y_test_k, y_pred_test_k)
mae_k = mean_absolute_error(y_test_k, y_pred_test_k)
mslr_k = mean_squared_log_error(y_test_k, y_pred_test_k)
#mrse_k = mean_relative_squared_error(y_test_k, y_pred_test_k)

eval_metrics_k = {
    'Eval_metrics': ['R2 Score', 'MSE', 'MAE', 'MSLE', 'Training time'],
    'Elastic-Net Model': [r2_k, mse_k, mae_k, mslr_k, training_time_k]
}

df_metrics_k = pd.DataFrame(eval_metrics_k)
df_metrics_k.to_csv('metrics_k.csv', index=False)

print(tabulate(df_metrics_k.round(4), headers='keys', tablefmt='pretty', showindex=False))


Epoch 1/1500


/home/chris/venv/cdd/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 - 1s - 164ms/step - loss: 43462.9766 - mean_absolute_error: 206.4867 - val_loss: 45712.4180 - val_mean_absolute_error: 211.5627 - learning_rate: 5.0000e-04
Epoch 2/1500
7/7 - 0s - 9ms/step - loss: 42755.7227 - mean_absolute_error: 204.8002 - val_loss: 44852.1172 - val_mean_absolute_error: 209.4339 - learning_rate: 5.0000e-04
Epoch 3/1500
7/7 - 0s - 7ms/step - loss: 41517.8828 - mean_absolute_error: 201.7410 - val_loss: 43315.6328 - val_mean_absolute_error: 205.5717 - learning_rate: 5.0000e-04
Epoch 4/1500
7/7 - 0s - 8ms/step - loss: 39427.1641 - mean_absolute_error: 196.4710 - val_loss: 40694.2109 - val_mean_absolute_error: 198.8100 - learning_rate: 5.0000e-04
Epoch 5/1500
7/7 - 0s - 8ms/step - loss: 35745.1836 - mean_absolute_error: 186.8752 - val_loss: 36524.9805 - val_mean_absolute_error: 187.5247 - learning_rate: 5.0000e-04
Epoch 6/1500
7/7 - 0s - 8ms/step - loss: 30161.0898 - mean_absolute_error: 170.9375 - val_loss: 30421.5352 - val_mean_absolute_error: 169.5954 - learning_ra

Model saved to 'keras_nn_model.h5'.
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/stepWARNING:tensorflow:5 out of the last 15 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7ea0a041b560> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
+---------------+-------------------+
| Eval_metrics  | Elastic-Net Model |
+---------------+-------------------+
|   R2 Score    |      -1.673       |
|      MSE      |     3687.8452     |
|      MAE      |      42.4433      |
|     MSLE      |      0.1118       |
| Training time |      5.7196       |
+---------------+-------------------+


A negative R² score greater than 1 indicates that the model is performing worse than a baseline model that predicts the mean of the target variable. This suggests that the model is not capturing the relationship between the input features and the target variable effectively. Below are steps to improve your Keras model:

1. Check Data Preprocessing
a. Scale Input Features
Ensure that all input features are properly scaled. You are already using StandardScaler, which is good. However, verify that the target variable (y_train_k and y_test_k) is on the correct scale.

b. Handle Outliers
Outliers in the input features or target variable can negatively impact the model's performance. Use techniques like z-score or IQR to detect and handle outliers.

c. Check Target Variable Distribution
If the target variable has a skewed distribution, apply a transformation like log, sqrt, or Box-Cox to make it more normal.

# Example: Log transformation for the target variable
y_train_k = np.log1p(y_train_k)
y_test_k = np.log1p(y_test_k)

2. Improve Model Architecture
a. Add More Layers and Neurons
Your current architecture might be too simple. Add more layers or increase the number of neurons in each layer.

model = Sequential([
    Dense(512, input_dim=x_train_k.shape[1], activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(1, activation='linear')
])

b. Use Better Activation Functions
While ReLU is a good default, you can experiment with other activation functions like ELU or LeakyReLU.

from tensorflow.keras.layers import LeakyReLU

model = Sequential([
    Dense(512, input_dim=x_train_k.shape[1]),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(256),
    LeakyReLU(alpha=0.1),
    Dropout(0.3),
    Dense(128),
    LeakyReLU(alpha=0.1),
    Dense(1, activation='linear')
])

3. Adjust Hyperparameters
a. Learning Rate
The learning rate might be too high or too low. Experiment with different values.

from tensorflow.keras.optimizers import Adam

model.compile(optimizer=Adam(learning_rate=0.0005), loss='mean_squared_error', metrics=['mean_absolute_error'])

b. Batch Size
A batch size of 5 might be too small. Try larger values like 16, 32, or 64.

training = model.fit(x_train_k, y_train_k, epochs=1500, validation_split=0.2, batch_size=32,
                     verbose=2, callbacks=[early_stop, reduce_lr])

c. Early Stopping and ReduceLROnPlateau
You are already using these callbacks, but you can fine-tune their parameters.

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

4. Evaluate Model Performance
a. Check for Overfitting
If the training loss is much lower than the validation loss, the model is overfitting. Use regularization techniques like Dropout or L2 regularization.

from tensorflow.keras.regularizers import l2

model = Sequential([
    Dense(512, input_dim=x_train_k.shape[1], activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(256, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(1, activation='linear')
])

b. Check for Underfitting
If both training and validation losses are high, the model is underfitting. Increase the complexity of the model by adding layers or neurons.

5. Use Cross-Validation
Instead of relying on a single train-test split, use cross-validation to ensure the model generalizes well.


kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_index, val_index in kf.split(x_train_k):
    x_train_fold, x_val_fold = x_train_k[train_index], x_train_k[val_index]
    y_train_fold, y_val_fold = y_train_k[train_index], y_train_k[val_index]
    model.fit(x_train_fold, y_train_fold, validation_data=(x_val_fold, y_val_fold), epochs=500, batch_size=32)


6. Debugging Negative R²
a. Check Predictions
Ensure that the predictions are on the same scale as the target variable. If you applied a transformation (e.g., log), reverse it before calculating metrics.

# Reverse log transformation
y_pred_test_k = np.expm1(y_pred_test_k)
y_test_k = np.expm1(y_test_k)

b. Check Data Leakage
Ensure that no information from the test set is leaking into the training set. For example, verify that the scaler is fit only on the training data.

7. Experiment with Advanced Techniques
a. Feature Engineering
Create new features or remove irrelevant ones to improve the model's ability to capture relationships.

b. Hyperparameter Tuning
Use libraries like Keras Tuner or Optuna to automate hyperparameter tuning.

from kerastuner.tuners import RandomSearch

def build_model(hp):
    model = Sequential([
        Dense(hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dropout(hp.Float('dropout', min_value=0.1, max_value=0.5, step=0.1)),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer=Adam(learning_rate=hp.Choice('learning_rate', [0.001, 0.0005, 0.0001])),
                  loss='mean_squared_error', metrics=['mean_absolute_error'])
    return model

tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
    directory='my_dir',
    project_name='keras_tuning'
)

tuner.search(x_train_k, y_train_k, epochs=100, validation_split=0.2)

Summary of Improvements
Preprocess the data: Scale inputs and transform the target variable if necessary.
Improve the architecture: Add layers, neurons, and regularization.
Tune hyperparameters: Adjust learning rate, batch size, and epochs.
Evaluate properly: Ensure predictions are on the same scale as the target.
Use cross-validation: Validate the model across multiple splits.
By implementing these steps, you should see an improvement in the R² score and overall model performance. Let me know if you'd like help with any specific part!

